# NetSec - Advanced Threat Detection (Companion Notebook)

A **standalone** analysis layer for the NetSec Wireshark-ML Dashboard. It runs on
**real Wireshark captures** (`.pcapng` / `.pcap`) through `tshark`, using the same
field-extraction approach as the main notebook - then adds six detection engines
focused on *encrypted-traffic-aware* and *behavioral* threats that the volumetric /
ML layer does not cover.

| # | Engine | Detects | MITRE ATT&CK |
|---|--------|---------|--------------|
| 1 | ARP / DHCP integrity | ARP spoofing (MITM), rogue DHCP server | T1557.002 / T1557 |
| 2 | DNS tunneling | Covert exfiltration / C2 over DNS | T1071.004 / T1048 |
| 3 | DGA | Algorithmically-generated C2 domains | T1568.002 |
| 4 | Beaconing | Periodic C2 check-ins (RITA-style regularity) | T1071 / T1571 |
| 5 | TLS fingerprinting | New/unusual clients, no-SNI-to-IP, SNI↔IP mismatch | T1071.001 / T1090 |
| 6 | Fusion + timeline | Correlates per-device signals into a kill-chain risk score | - |

**How to use**
1. Install Wireshark so `tshark` is on your PATH.
2. `pip install pandas numpy plotly`.
3. Edit `PCAP_FILES` in the **Config** cell to point at your real captures.
4. **Run All**. Findings appear as tables + figures and are exported to CSV/JSON.

**Reference data (optional):** drop the project's `cloud_ranges.json` next to this
notebook to enrich the TLS SNI↔IP check with provider attribution.

> The notebook is **read-only** toward the network - it only parses capture files. It
> never transmits, scans, or injects. Run it only on traffic you are authorized to analyze.

In [ ]:
# --- Section 1 - Imports ---
import os, io, re, math, json, shutil, subprocess, ipaddress, collections
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)
print("imports ok - pandas", pd.__version__, "| numpy", np.__version__)

## Section 2 - Configuration

Point `PCAP_FILES` at your real captures. Thresholds are conservative defaults -
tune them to your network's baseline (a noisy IoT network needs higher beacon/DNS
thresholds than a quiet office).

In [ ]:
# --- Section 2 - Config ---

# >>> EDIT: your real capture files. Use raw strings on Windows: r"C:\\caps\\s1.pcapng"
PCAP_FILES = [
    # ("S1", "/path/to/session1.pcapng"),
    # ("S2", "/path/to/session2.pcapng"),
]

def _find_cfg(name):
    for c in [Path.cwd()/name, Path.cwd()/"app"/name, Path.cwd().parent/name,
              Path.cwd().parent/"app"/name, Path.home()/name]:
        if c.exists(): return str(c)
    return name
CLOUD_RANGES_JSON = _find_cfg("cloud_ranges.json")   # optional, enriches TLS check if present

# Detection thresholds (tune to your environment)
BEACON_MIN_EVENTS  = 16     # min connections for a (src,dst) pair to be scored
BEACON_SCORE_FLAG  = 0.80   # 0..1 regularity score above which we flag
DNS_UNIQUE_MIN     = 20     # min unique subdomains under one apex to consider tunneling
DNS_UNIQUE_RATIO   = 0.90   # unique/total query ratio (tunneling needs ~1.0, no caching)
DNS_ENTROPY_FLAG   = 3.8    # Shannon entropy of leftmost label
DNS_LABEL_LEN_FLAG = 40     # mean full-qname length
NX_STORM_MIN       = 30     # NXDOMAIN responses per device to flag a storm
DGA_MIN_LABEL_LEN  = 7      # ignore short labels for DGA scoring
DGA_LOGPROB_FLAG   = None   # None => auto threshold = mean - 1*std of observed labels
FUSION_WINDOW_MIN  = 15     # minutes; window for kill-chain correlation

_RFC1918 = [ipaddress.ip_network(n) for n in ("10.0.0.0/8","172.16.0.0/12","192.168.0.0/16")]
def is_private(ip):
    # True for local / non-routable / multicast / loopback addresses
    if not ip: return False
    try: a = ipaddress.ip_address(ip)
    except Exception: return False
    return a.is_private or a.is_link_local or a.is_loopback or a.is_multicast or a.is_unspecified

print("config loaded -", len(PCAP_FILES), "capture(s) configured")

## Section 3 - PCAP Loader (tshark)

Extracts a unified per-packet table with the same fields as the main dashboard, plus
TLS (`ja3`/`ja4`/SNI), ARP opcode/targets, DNS rcode, and the DHCP server id. Unknown
fields on older tshark builds simply come back empty (no crash). One DataFrame, one schema,
fed to every detector.

In [ ]:
# --- Section 3 - Loader ---
TSHARK_FIELDS = [
    "frame.time_epoch", "frame.len",
    "eth.src", "eth.dst",
    "ip.src", "ip.dst",
    "_ws.col.Protocol",
    "tcp.srcport", "tcp.dstport", "tcp.flags",
    "udp.srcport", "udp.dstport",
    "dns.qry.name", "dns.qry.type", "dns.flags.rcode", "dns.flags.response",
    "arp.opcode", "arp.src.proto_ipv4", "arp.src.hw_mac", "arp.dst.proto_ipv4",
    "tls.handshake.extensions_server_name", "tls.handshake.ja3", "tls.handshake.ja4",
    "dhcp.option.dhcp_server_id",
]
COLS = [
    "ts", "len", "eth_src", "eth_dst", "ip_src", "ip_dst", "proto",
    "tcp_sport", "tcp_dport", "tcp_flags", "udp_sport", "udp_dport",
    "dns_qname", "dns_qtype", "dns_rcode", "dns_response",
    "arp_opcode", "arp_psrc", "arp_hwsrc", "arp_pdst",
    "tls_sni", "tls_ja3", "tls_ja4", "dhcp_sid",
]

def find_tshark():
    c = shutil.which("tshark")
    if c: return c
    for p in ["/usr/bin/tshark", "/usr/local/bin/tshark",
              "/Applications/Wireshark.app/Contents/MacOS/tshark",
              r"C:\Program Files\Wireshark\tshark.exe",
              r"C:\Program Files (x86)\Wireshark\tshark.exe"]:
        if os.path.exists(p): return p
    return None

TSHARK = find_tshark()
print("tshark:", TSHARK or "NOT FOUND - install Wireshark CLI and re-run")

def load_capture(path, label):
    if not TSHARK: raise RuntimeError("tshark not found on PATH")
    if not os.path.exists(path): raise FileNotFoundError(path)
    cmd = [TSHARK, "-r", str(path), "-n", "-T", "fields",
           "-E", "header=n", "-E", "separator=\t", "-E", "occurrence=f", "-E", "quote=n"]
    for f in TSHARK_FIELDS: cmd += ["-e", f]
    raw = subprocess.check_output(cmd, encoding="utf-8", errors="replace", stderr=subprocess.DEVNULL)
    df = pd.read_csv(io.StringIO(raw), sep="\t", header=None, names=COLS,
                     dtype=str, na_filter=False, low_memory=False)
    if not len(df): raise RuntimeError("tshark parsed 0 rows")
    df["ts"]  = pd.to_numeric(df["ts"], errors="coerce")
    df["len"] = pd.to_numeric(df["len"], errors="coerce").fillna(0).astype(int)
    df = df.dropna(subset=["ts"]).reset_index(drop=True)
    df["session"] = label
    return df

frames = []
for label, path in PCAP_FILES:
    try:
        d = load_capture(path, label); frames.append(d)
        print(f"  {label}: {len(d):,} packets  <-  {path}")
    except Exception as e:
        print(f"  [!] {label} failed: {e}")

if frames:
    PK = pd.concat(frames, ignore_index=True)
else:
    PK = pd.DataFrame({c: pd.Series(dtype="object") for c in COLS + ["session"]})
    PK["ts"] = pd.to_numeric(PK["ts"], errors="coerce")
    print("\n  No captures loaded yet - set PCAP_FILES in Section 2 and Run All.")

print(f"\nTotal: {len(PK):,} packets across {len(frames)} session(s)")

## Section 4 - Shared Helpers

Math + lookups used across detectors: Shannon entropy, registrable-domain extraction,
a character bigram language model for DGA scoring, the RITA-style beaconing scorer, the
`cloud_ranges.json` provider lookup, the MITRE technique map, and the uniform **signal**
record every detector emits.

In [ ]:
# --- Section 4 - Helpers ---

# Uniform signal schema emitted by every detector
SIGNAL_COLS = ["device","peer","signal","tactic","technique","score","severity",
               "count","first_ts","last_ts","detail"]
def sig(**kw):
    return {k: kw.get(k) for k in SIGNAL_COLS}

def shannon(s):
    if not s: return 0.0
    n = len(s)
    return -sum((c/n)*math.log2(c/n) for c in collections.Counter(s).values())

def vowel_ratio(s):
    s = s.lower()
    return (sum(c in "aeiou" for c in s)/len(s)) if s else 0.0

# Registrable ("apex") domain, with a small known multi-part-SLD table
_MULTI_SLD = {"co.uk","org.uk","ac.uk","gov.uk","co.il","org.il","ac.il","net.il","gov.il",
              "com.au","net.au","org.au","co.jp","com.br","co.in","com.cn","co.kr","co.za"}
def registrable(name):
    name = (name or "").strip(".").lower()
    parts = name.split(".")
    if len(parts) < 2: return name
    if ".".join(parts[-2:]) in _MULTI_SLD and len(parts) >= 3:
        return ".".join(parts[-3:])
    return ".".join(parts[-2:])
def leftmost_label(name):
    name = (name or "").strip(".")
    return name.split(".")[0] if name else ""

# Character bigram model: log-likelihood per char; English-like => high, DGA => low
_BG_VOCAB = "abcdefghijklmnopqrstuvwxyz0123456789-_."
def train_bigram(labels):
    cnt = collections.defaultdict(lambda: collections.defaultdict(int))
    for lab in labels:
        t = "^" + (lab or "").lower() + "$"
        for a, b in zip(t, t[1:]):
            cnt[a][b] += 1
    V = len(_BG_VOCAB) + 2
    model = {}
    for a, nxt in cnt.items():
        tot = sum(nxt.values()) + V
        row = {b: math.log((c+1)/tot) for b, c in nxt.items()}
        row["__default__"] = math.log(1/tot)
        model[a] = row
    model["__global_default__"] = math.log(1.0/V)
    return model
def score_label(model, lab):
    t = "^" + (lab or "").lower() + "$"; lp = 0.0; n = 0
    for a, b in zip(t, t[1:]):
        row = model.get(a)
        lp += (model["__global_default__"] if row is None else row.get(b, row["__default__"]))
        n += 1
    return lp/max(n, 1)

# Small benign label corpus to stabilise the DGA model when a capture is tiny.
# This is REFERENCE DATA (like cloud_ranges.json), not generated traffic.
COMMON_DOMAINS = ["google","youtube","facebook","amazon","microsoft","apple","netflix",
    "instagram","whatsapp","wikipedia","linkedin","github","cloudflare","akamai","fastly",
    "office","windowsupdate","icloud","gmail","outlook","spotify","twitch","reddit","yahoo",
    "bing","dropbox","adobe","zoom","slack","tiktok","snapchat","pinterest","ebay","paypal",
    "samsung","intel","nvidia","mozilla","ubuntu","debian","android","googleapis","gstatic",
    "doubleclick","cdn","edgekey","edgesuite","amazonaws","azure","digitalocean"]

def beacon_scores(ts, sizes):
    ts = np.sort(np.asarray(ts, dtype=float))
    if len(ts) < 3: return None
    d = np.diff(ts); d = d[d >= 0]
    if len(d) < 2: return None
    q1, q2, q3 = np.percentile(d, [25, 50, 75])
    iqr = q3 - q1
    skew = 1.0 if iqr == 0 else max(0.0, 1 - abs((q3 + q1 - 2*q2)/iqr))   # Bowley regularity
    mad  = np.median(np.abs(d - q2))
    disp = 1.0 if q2 == 0 else max(0.0, 1 - min(mad/q2, 1))               # interval tightness
    s = np.asarray(sizes, dtype=float)
    smed = np.median(s); smad = np.median(np.abs(s - smed))
    size = 1.0 if smed == 0 else max(0.0, 1 - min(smad/smed, 1))          # payload uniformity
    return dict(score=(skew+disp+size)/3, median_interval=float(q2),
                n=int(len(ts)), skew=skew, disp=disp, size=size)

def max_distinct_in_window(times, keys, window_s):
    # Two-pointer: max number of distinct keys co-occurring within any window_s span.
    if len(times) == 0: return 0
    order = np.argsort(times); t = np.asarray(times)[order]; k = np.asarray(keys)[order]
    cnt = collections.defaultdict(int); distinct = 0; best = 0; left = 0
    for right in range(len(t)):
        if cnt[k[right]] == 0: distinct += 1
        cnt[k[right]] += 1
        while t[right] - t[left] > window_s:
            cnt[k[left]] -= 1
            if cnt[k[left]] == 0: distinct -= 1
            left += 1
        best = max(best, distinct)
    return best

class CloudDB:
    # Provider lookup from cloud_ranges.json (static_ips + cidr_ranges + rdns_patterns)
    def __init__(self, path):
        self.static = {}; self.cidrs = []; self.rdns = []; self.ok = False; self.err = None
        try:
            d = json.load(open(path, encoding="utf-8"))
            self.static = d.get("static_ips", {}) or {}
            for e in d.get("cidr_ranges", []):
                try: self.cidrs.append((ipaddress.ip_network(e["cidr"]), e))
                except Exception: pass
            for e in d.get("rdns_patterns", []):
                try: self.rdns.append((re.compile(e["pattern"]), e))
                except Exception: pass
            self.ok = True
        except Exception as ex:
            self.err = str(ex)
    def by_ip(self, ip):
        e = self.static.get(ip)
        if e: return e.get("provider")
        try: a = ipaddress.ip_address(ip)
        except Exception: return None
        for net, e in self.cidrs:
            if a in net: return e.get("provider")
        return None
    def by_host(self, host):
        for rx, e in self.rdns:
            if rx.search(host or ""): return e.get("provider")
        return None

CLOUD = CloudDB(CLOUD_RANGES_JSON)
print("helpers ready | cloud_ranges:",
      ("ok (%d cidrs, %d rdns)" % (len(CLOUD.cidrs), len(CLOUD.rdns))) if CLOUD.ok else f"not loaded ({CLOUD.err})")

## Section 5 - Engine 1: ARP Spoofing & Rogue DHCP  ·  `T1557`

ARP has no authentication, so a MITM attacker forges replies binding the **gateway IP to
their own MAC**. Tell-tale signs in the capture:

- **One IP claimed by multiple MACs** (excluding legitimate DHCP re-assignment).
- **One MAC announcing many IPs** (the attacker impersonating victims + gateway).
- **Unsolicited / gratuitous ARP replies** far exceeding requests.
- **Rogue DHCP:** more than one DHCP server id handing out leases.

In [ ]:
# --- Section 5 - ARP / DHCP ---
def detect_arp_dhcp(PK):
    rows = []
    if not len(PK): return pd.DataFrame(rows, columns=SIGNAL_COLS)
    arp = PK[PK["arp_hwsrc"] != ""]
    bad_mac = {"", "00:00:00:00:00:00", "ff:ff:ff:ff:ff:ff"}

    if len(arp):
        a = arp[arp["arp_psrc"] != ""]
        # IP -> set(MAC)
        for ip, grp in a.groupby("arp_psrc"):
            macs = {m for m in grp["arp_hwsrc"].unique() if m not in bad_mac}
            if ip in ("0.0.0.0", "") or len(macs) <= 1: continue
            rows.append(sig(device=ip, peer=";".join(sorted(macs)), signal="arp_ip_multi_mac",
                tactic="Collection / MITM", technique="T1557.002",
                score=min(1.0, 0.6 + 0.15*len(macs)), severity="high", count=len(grp),
                first_ts=grp["ts"].min(), last_ts=grp["ts"].max(),
                detail=f"IP {ip} claimed by {len(macs)} MACs: {sorted(macs)}"))
        # MAC -> set(IP)
        for mac, grp in a.groupby("arp_hwsrc"):
            if mac in bad_mac: continue
            ips = {i for i in grp["arp_psrc"].unique() if i not in ("0.0.0.0", "")}
            if len(ips) < 4: continue
            rows.append(sig(device=mac, peer=";".join(sorted(ips)[:8]), signal="arp_mac_many_ips",
                tactic="Collection / MITM", technique="T1557.002",
                score=min(1.0, 0.5 + 0.08*len(ips)), severity="high", count=len(grp),
                first_ts=grp["ts"].min(), last_ts=grp["ts"].max(),
                detail=f"MAC {mac} announced {len(ips)} distinct IPs"))
        # Unsolicited replies (opcode 2) >> requests (opcode 1)
        rep = arp[arp["arp_opcode"] == "2"]; req = arp[arp["arp_opcode"] == "1"]
        rc = rep.groupby("arp_hwsrc").size(); qc = req.groupby("arp_hwsrc").size()
        for mac, n in rc.items():
            if mac in bad_mac: continue
            q = int(qc.get(mac, 0))
            if n >= 10 and n > 3*max(q, 1):
                sub = rep[rep["arp_hwsrc"] == mac]
                rows.append(sig(device=mac, peer="", signal="arp_gratuitous_flood",
                    tactic="Collection / MITM", technique="T1557.002",
                    score=min(1.0, 0.4 + n/120.0), severity="medium", count=int(n),
                    first_ts=sub["ts"].min(), last_ts=sub["ts"].max(),
                    detail=f"{n} ARP replies vs {q} requests (unsolicited)"))

    # Rogue DHCP
    srv = [s for s in PK["dhcp_sid"].unique() if s] if "dhcp_sid" in PK.columns else []
    if len(srv) > 1:
        for s in srv:
            sub = PK[PK["dhcp_sid"] == s]
            rows.append(sig(device=s, peer=";".join(srv), signal="rogue_dhcp",
                tactic="Collection / MITM", technique="T1557",
                score=0.7, severity="high", count=len(sub),
                first_ts=sub["ts"].min(), last_ts=sub["ts"].max(),
                detail=f"{len(srv)} DHCP servers offering leases: {srv}"))
    return pd.DataFrame(rows, columns=SIGNAL_COLS)
print("engine 1 ready")

## Section 6 - Engine 2: DNS Tunneling & NXDOMAIN  ·  `T1071.004`

DNS exfiltration encodes data into subdomain labels of an attacker-controlled domain.
Its structural weakness is **uniqueness**: every query must differ or caching kills the
channel. We flag an apex domain with **many unique, high-entropy, long** subdomains and a
near-1.0 unique-to-total ratio - and separately flag **NXDOMAIN storms** per device
(failed resolutions, a shared symptom of tunneling and DGA).

In [ ]:
# --- Section 6 - DNS tunneling ---
def detect_dns_tunnel(PK):
    rows = []
    if not len(PK) or "dns_qname" not in PK.columns:
        return pd.DataFrame(rows, columns=SIGNAL_COLS)
    q = PK[(PK["dns_qname"] != "") & (PK["dns_response"] != "1")].copy()
    if len(q):
        q["reg"]  = q["dns_qname"].map(registrable)
        q["lbl"]  = q["dns_qname"].map(leftmost_label)
        q["ent"]  = q["lbl"].map(shannon)
        q["qlen"] = q["dns_qname"].str.len()
        for reg, g in q.groupby("reg"):
            if not reg: continue
            total = len(g); uniq = g["dns_qname"].nunique(); ratio = uniq/max(total, 1)
            max_ent = float(g["ent"].max()); mean_len = float(g["qlen"].mean())
            if uniq >= DNS_UNIQUE_MIN and ratio >= DNS_UNIQUE_RATIO and \
               (max_ent >= DNS_ENTROPY_FLAG or mean_len >= DNS_LABEL_LEN_FLAG):
                asker = g["ip_src"].value_counts()
                dev = asker.index[0] if len(asker) else ""
                score = min(1.0, 0.4 + 0.3*(max_ent/4.5) + 0.3*min(uniq/200.0, 1))
                rows.append(sig(device=dev, peer=reg, signal="dns_tunneling",
                    tactic="Exfiltration / C2", technique="T1071.004",
                    score=score, severity="high", count=int(uniq),
                    first_ts=g["ts"].min(), last_ts=g["ts"].max(),
                    detail=f"{uniq} unique subdomains under {reg} "
                           f"(uniq_ratio={ratio:.2f}, max_entropy={max_ent:.2f}, mean_len={mean_len:.0f})"))
    # NXDOMAIN storms (response rcode==3; asker is ip_dst on the reply)
    r = PK[(PK["dns_qname"] != "") & (PK["dns_response"] == "1") & (PK["dns_rcode"] == "3")]
    if len(r):
        for dev, g in r.groupby("ip_dst"):
            if dev == "" or len(g) < NX_STORM_MIN: continue
            rows.append(sig(device=dev, peer="", signal="nxdomain_storm",
                tactic="Command and Control", technique="T1568.002",
                score=min(1.0, 0.3 + len(g)/300.0), severity="medium", count=len(g),
                first_ts=g["ts"].min(), last_ts=g["ts"].max(),
                detail=f"{len(g)} NXDOMAIN responses (failed lookups - DGA/tunnel symptom)"))
    return pd.DataFrame(rows, columns=SIGNAL_COLS)
print("engine 2 ready")

## Section 7 - Engine 3: DGA Domains  ·  `T1568.002`

Malware that generates pseudo-random C2 domains produces labels that don't look like
language. We train a character **bigram model on the benign domains that actually
resolved in *this* capture** (NOERROR), augmented by a small reference list, then score
every observed apex label. Low log-likelihood **plus** high entropy or low vowel ratio →
DGA candidate. Pairing this with the NXDOMAIN storm and beaconing signals (fused in
Section 9) is what turns a guess into a kill-chain.

In [ ]:
# --- Section 7 - DGA ---
def detect_dga(PK):
    rows = []
    if not len(PK) or "dns_qname" not in PK.columns:
        return pd.DataFrame(rows, columns=SIGNAL_COLS)
    q = PK[(PK["dns_qname"] != "") & (PK["dns_response"] != "1")]
    if not len(q): return pd.DataFrame(rows, columns=SIGNAL_COLS)

    # Benign baseline = apex labels that resolved with NOERROR in this capture
    resolved = PK[(PK["dns_response"] == "1") & (PK["dns_rcode"] == "0") & (PK["dns_qname"] != "")]
    base = [registrable(x).split(".")[0] for x in resolved["dns_qname"].unique()]
    base = [b for b in base if b and len(b) >= 3]
    if len(set(base)) < 30:
        base = base + COMMON_DOMAINS
    model = train_bigram(base)

    labels = {registrable(x).split(".")[0] for x in q["dns_qname"].unique()}
    scored = [(lab, score_label(model, lab), shannon(lab), vowel_ratio(lab))
              for lab in labels if lab and len(lab) >= DGA_MIN_LABEL_LEN]
    if not scored: return pd.DataFrame(rows, columns=SIGNAL_COLS)

    arr = np.array([s[1] for s in scored], dtype=float)
    thr = DGA_LOGPROB_FLAG if DGA_LOGPROB_FLAG is not None else float(arr.mean() - arr.std())
    qreg = q.assign(_lab=q["dns_qname"].map(lambda x: registrable(x).split(".")[0]))
    for lab, lp, ent, vw in scored:
        if lp < thr and (ent >= 3.2 or vw < 0.25):
            full = qreg[qreg["_lab"] == lab]
            asker = full["ip_src"].value_counts()
            dev = asker.index[0] if len(asker) else ""
            score = min(1.0, 0.4 + min(thr-lp, 3.0)/6.0 + (0.2 if vw < 0.25 else 0.0))
            rows.append(sig(device=dev, peer=lab, signal="dga_domain",
                tactic="Command and Control", technique="T1568.002",
                score=score, severity="medium", count=int(full["dns_qname"].nunique()),
                first_ts=full["ts"].min(), last_ts=full["ts"].max(),
                detail=f"label '{lab}' logprob={lp:.2f} (thr={thr:.2f}), entropy={ent:.2f}, vowel_ratio={vw:.2f}"))
    return pd.DataFrame(rows, columns=SIGNAL_COLS)
print("engine 3 ready")

## Section 8 - Engine 4: Beaconing  ·  `T1071` / `T1571`

C2 implants check in on a near-fixed cadence; human traffic is bursty. For each
local→external `(src,dst)` pair we score three regularity dimensions on the connection
timestamps and packet sizes - **Bowley skew** (interval symmetry), **MAD dispersion**
(interval tightness), and **payload uniformity** - and flag a high combined score.
UDP/123 (NTP) is skipped as a known-periodic service.

In [ ]:
# --- Section 8 - Beaconing ---
def detect_beaconing(PK):
    rows = []
    if not len(PK): return pd.DataFrame(rows, columns=SIGNAL_COLS)
    df = PK[(PK["ip_src"] != "") & (PK["ip_dst"] != "")]
    for (src, dst), g in df.groupby(["ip_src", "ip_dst"]):
        if len(g) < BEACON_MIN_EVENTS: continue
        if not is_private(src) or is_private(dst): continue        # local -> external only
        dports = set(g["tcp_dport"]) | set(g["udp_dport"])
        if "123" in dports: continue                               # NTP
        r = beacon_scores(g["ts"].values, g["len"].values)
        if not r or r["median_interval"] < 1: continue
        if r["score"] >= BEACON_SCORE_FLAG:
            rows.append(sig(device=src, peer=dst, signal="beaconing",
                tactic="Command and Control", technique="T1071",
                score=r["score"], severity=("high" if r["score"] >= 0.9 else "medium"),
                count=r["n"], first_ts=g["ts"].min(), last_ts=g["ts"].max(),
                detail=f"{r['n']} conns, median interval {r['median_interval']:.1f}s, "
                       f"regularity {r['score']:.2f} (skew {r['skew']:.2f} / disp {r['disp']:.2f} / size {r['size']:.2f})"))
    return pd.DataFrame(rows, columns=SIGNAL_COLS)
print("engine 4 ready")

## Section 9 - Engine 5: TLS Fingerprinting (JA3/JA4 + SNI↔IP)  ·  `T1071.001` / `T1090`

The TLS ClientHello is sent in clear and its JA3/JA4 hash fingerprints the *client
software*. We surface:

- **Rare JA3** - a fingerprint seen on a single device a handful of times (a new/unusual client).
- **TLS to an external IP with no SNI** - a classic direct-to-IP C2 pattern.
- **SNI↔IP provider mismatch** - SNI claims one provider but the destination IP belongs to
  another (possible domain fronting), resolved via `cloud_ranges.json`.

In [ ]:
# --- Section 9 - TLS ---
def detect_tls(PK, cloud=None):
    rows = []
    if not len(PK) or "tls_ja3" not in PK.columns:
        return pd.DataFrame(rows, columns=SIGNAL_COLS)
    tls = PK[PK["tls_ja3"] != ""]
    if len(tls):
        ndev = tls.groupby("tls_ja3")["ip_src"].nunique()
        ncnt = tls.groupby("tls_ja3").size()
        for ja3, nd in ndev.items():
            if nd == 1 and int(ncnt[ja3]) <= 3:
                sub = tls[tls["tls_ja3"] == ja3]; dev = sub["ip_src"].iloc[0]
                rows.append(sig(device=dev, peer=ja3[:16], signal="rare_ja3",
                    tactic="Command and Control", technique="T1071.001",
                    score=0.5, severity="low", count=int(ncnt[ja3]),
                    first_ts=sub["ts"].min(), last_ts=sub["ts"].max(),
                    detail=f"JA3 {ja3} seen {int(ncnt[ja3])}x on a single device {dev} (new/unusual client)"))
    # TLS handshakes to external IPs with no SNI
    hs = PK[(PK["tls_ja3"] != "") | (PK["tls_sni"] != "")]
    nosni = hs[(hs["tls_sni"] == "") & (hs["ip_dst"] != "")]
    nosni = nosni[~nosni["ip_dst"].map(is_private)]
    if len(nosni):
        for dev, g in nosni.groupby("ip_src"):
            if dev == "": continue
            rows.append(sig(device=dev, peer=";".join(sorted(g["ip_dst"].unique())[:5]),
                signal="tls_no_sni_external", tactic="Command and Control", technique="T1071.001",
                score=0.45, severity="low", count=len(g),
                first_ts=g["ts"].min(), last_ts=g["ts"].max(),
                detail=f"{len(g)} TLS handshakes to external IPs without SNI"))
    # SNI vs destination-IP provider mismatch
    if cloud and cloud.ok:
        m = PK[(PK["tls_sni"] != "") & (PK["ip_dst"] != "")]
        m = m[~m["ip_dst"].map(is_private)].drop_duplicates(["ip_src","ip_dst","tls_sni"])
        for _, r in m.iterrows():
            ps = cloud.by_host(r["tls_sni"]); pi = cloud.by_ip(r["ip_dst"])
            if ps and pi and ps != pi:
                rows.append(sig(device=r["ip_src"], peer=r["tls_sni"], signal="sni_ip_mismatch",
                    tactic="Command and Control", technique="T1090",
                    score=0.6, severity="medium", count=1,
                    first_ts=r["ts"], last_ts=r["ts"],
                    detail=f"SNI {r['tls_sni']} -> {ps}, but dst {r['ip_dst']} -> {pi} (possible domain fronting)"))
    return pd.DataFrame(rows, columns=SIGNAL_COLS)
print("engine 5 ready")

## Section 10 - Engine 6: Fusion & Kill-Chain Risk

Individual signals are noisy; *correlated* signals are not. We concatenate every detector's
output and, per device, apply a **kill-chain boost**: the more *distinct techniques* co-occur
within a `FUSION_WINDOW_MIN` window, the higher the multiplier. An NXDOMAIN storm → a DGA
hit → beaconing, all in 15 minutes, ranks far above any one alone.

In [ ]:
# --- Section 10 - Fusion + run all ---
def fuse(signals):
    if not len(signals):
        return signals, pd.DataFrame(columns=["device","signals","signal_types","max_score",
                                              "kill_chain_boost","risk","techniques","detail"])
    s = signals.copy()
    s["first_ts"] = pd.to_numeric(s["first_ts"], errors="coerce")
    win = FUSION_WINDOW_MIN*60
    out = []
    for dev, g in s.groupby("device"):
        base = float(g["score"].max())
        t = g["first_ts"].dropna().values
        best = max_distinct_in_window(t, g.loc[g["first_ts"].notna(), "technique"].values, win) if len(t) else 1
        boost = 1.0 + 0.5*max(best-1, 0)
        out.append(dict(device=dev, signals=len(g), signal_types=int(g["signal"].nunique()),
            max_score=round(base, 3), kill_chain_boost=round(boost, 2),
            risk=round(min(1.0, base*boost), 3),
            techniques=";".join(sorted(g["technique"].dropna().unique())),
            detail="; ".join(list(dict.fromkeys(g["signal"]))[:6])))
    dev = pd.DataFrame(out).sort_values("risk", ascending=False).reset_index(drop=True)
    return s, dev

SIGNALS = pd.concat([
    detect_arp_dhcp(PK),
    detect_dns_tunnel(PK),
    detect_dga(PK),
    detect_beaconing(PK),
    detect_tls(PK, CLOUD),
], ignore_index=True)
SIGNALS, DEVICE_RISK = fuse(SIGNALS)

n_dev = SIGNALS["device"].nunique() if len(SIGNALS) else 0
print(f"{len(SIGNALS)} signal(s) across {n_dev} device(s)")
if len(SIGNALS):
    print("\nTop devices by kill-chain risk:")
    try: display(DEVICE_RISK.head(20))
    except NameError: print(DEVICE_RISK.head(20).to_string())
    print("\nTop signals:")
    top = SIGNALS.sort_values("score", ascending=False).head(30)
    try: display(top)
    except NameError: print(top.to_string())
else:
    print("No signals - load real captures in Section 2 (and ensure tshark is installed).")

## Section 11 - Visualizations

In [ ]:
# --- Section 11 - Plots ---
def plot_device_risk(DEVICE_RISK):
    if not len(DEVICE_RISK):
        print("no device risk to plot"); return None
    d = DEVICE_RISK.head(15).iloc[::-1]
    fig = go.Figure(go.Bar(x=d["risk"], y=d["device"], orientation="h",
        text=d["techniques"], marker=dict(color=d["risk"], colorscale="Reds", cmin=0, cmax=1)))
    fig.update_layout(title="Per-device kill-chain risk (top 15)",
        xaxis_title="risk (0..1)", yaxis_title="device", height=520, template="plotly_white")
    return fig

def plot_attack_timeline(SIGNALS):
    if not len(SIGNALS):
        print("no signals to plot"); return None
    s = SIGNALS.copy()
    s["t"] = pd.to_datetime(pd.to_numeric(s["first_ts"], errors="coerce"), unit="s")
    s = s.dropna(subset=["t"])
    if not len(s):
        print("signals have no timestamps to plot"); return None
    fig = px.scatter(s, x="t", y="device", color="technique", size="score",
        hover_data=["signal","peer","severity","detail"],
        title="ATT&CK signal timeline (per device)")
    fig.update_layout(height=560, template="plotly_white", xaxis_title="time", yaxis_title="device")
    return fig

f1 = plot_device_risk(DEVICE_RISK)
if f1: f1.show()
f2 = plot_attack_timeline(SIGNALS)
if f2: f2.show()

## Section 12 - Export Findings

Writes machine-readable findings next to the notebook for ticketing / SIEM ingest.

In [ ]:
# --- Section 12 - Export ---
OUT_CSV  = "threat_findings.csv"
OUT_DEV  = "device_risk.csv"
OUT_JSON = "threat_findings.json"
if len(SIGNALS):
    SIGNALS.to_csv(OUT_CSV, index=False)
    DEVICE_RISK.to_csv(OUT_DEV, index=False)
    json.dump({"generated": datetime.utcnow().isoformat()+"Z",
               "signals": SIGNALS.to_dict(orient="records"),
               "device_risk": DEVICE_RISK.to_dict(orient="records")},
              open(OUT_JSON, "w", encoding="utf-8"), ensure_ascii=False, indent=2, default=str)
    print(f"wrote {OUT_CSV}, {OUT_DEV}, {OUT_JSON}  ({len(SIGNALS)} signals)")
else:
    print("nothing to export yet - load captures and Run All")